# Chapter 3. QNode와 디바이스

**Quantum AI: 핸즈온 양자머신러닝 with PennyLane — Part 1**

## 학습 목표
- QNode가 어떻게 회로와 디바이스를 결합하는지 이해
- 시뮬레이터 종류와 특성 비교 (`default.qubit`, `lightning.qubit`, `default.mixed`)
- shot 기반 측정과 analytic 측정의 차이 체감
- 같은 회로에서 측정 종류만 바꿔보기

In [7]:
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt
import time

## 3.1 QNode 객체 직접 만들기
데코레이터 없이도 QNode를 만들 수 있다. 함수와 디바이스를 직접 결합하는 방식이다.

In [8]:
dev = qml.device('default.qubit', wires=2)

def circuit_fn(theta):
    qml.RY(theta, wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.expval(qml.PauliZ(1))

# QNode 객체 생성
qnode = qml.QNode(circuit_fn, dev)
print(qnode(0.5))
print('타입:', type(qnode).__name__)

0.8775825618903726
타입: QNode


## 3.2 디바이스 비교
동일한 회로를 세 종류의 시뮬레이터에서 돌려본다.
- `default.qubit`: 파이썬 기반, 가장 일반적
- `lightning.qubit`: C++ 백엔드, 큰 회로에서 빠름
- `default.mixed`: 혼합 상태 시뮬레이션 (노이즈 모델링 가능)

In [9]:
device_names = ['default.qubit', 'lightning.qubit', 'default.mixed']
results = {}

for name in device_names:
    dev_x = qml.device(name, wires=4)
    @qml.qnode(dev_x)
    def circ(angles):
        qml.AngleEmbedding(angles, wires=range(4))
        qml.BasicEntanglerLayers(
            np.random.RandomState(0).uniform(0, 2 * np.pi, (3, 4)),
            wires=range(4),
        )
        return qml.expval(qml.PauliZ(0))
    angles = np.linspace(0, 1, 4)
    start = time.perf_counter()
    val = circ(angles)
    elapsed = time.perf_counter() - start
    results[name] = (val, elapsed)

for name, (val, t) in results.items():
    print(f'{name:20s}: <Z_0> = {val:+.6f}  ({t*1000:.2f} ms)')

default.qubit       : <Z_0> = -0.221709  (6.83 ms)
lightning.qubit     : <Z_0> = -0.221709  (2.42 ms)
default.mixed       : <Z_0> = -0.221709  (10.59 ms)


세 디바이스 모두 동일한 기댓값을 반환한다 (수치 오차 범위 내). `lightning.qubit`은 큐비트 수가 많아질수록 격차가 벌어진다.

## 3.3 Analytic vs Shot-based 측정
`shots=None`이면 진폭에서 정확한 기댓값을 계산한다 (analytic). `shots=N`을 지정하면 실제 양자 하드웨어처럼 N번 측정해서 평균을 낸다.

In [10]:
# Analytic 측정
dev_analytic = qml.device('default.qubit', wires=1, shots=None)

@qml.qnode(dev_analytic)
def measure_analytic():
    qml.Hadamard(wires=0)
    qml.RY(0.3, wires=0)
    return qml.expval(qml.PauliZ(0))

print(f'Analytic 기댓값: {measure_analytic():.6f}')

# Shot-based 측정 - 여러 shot 수로 비교
shot_counts = [10, 100, 1000, 10000]
for n_shots in shot_counts:
    dev_shot = qml.device('default.qubit', wires=1, shots=n_shots)
    @qml.qnode(dev_shot)
    def measure_shot():
        qml.Hadamard(wires=0)
        qml.RY(0.3, wires=0)
        return qml.expval(qml.PauliZ(0))
    estimates = np.array([measure_shot() for _ in range(20)])
    print(f'shots={n_shots:>5d}: 평균 {estimates.mean():+.4f}, '
          f'표준편차 {estimates.std():.4f}')

Analytic 기댓값: -0.295520
shots=   10: 평균 -0.3300, 표준편차 0.3537
shots=  100: 평균 -0.2990, 표준편차 0.1055
shots= 1000: 평균 -0.2918, 표준편차 0.0310
shots=10000: 평균 -0.2987, 표준편차 0.0091


shot 수가 많을수록 estimate의 분산이 작아진다 (1/√N 스케일). 실제 양자 하드웨어는 항상 shot 기반이므로, 이 노이즈를 다루는 법이 QML의 중요한 주제다.

## 3.4 측정 종류별 출력 비교
같은 회로의 출력을 세 가지 측정 방식으로 비교한다.

In [11]:
dev = qml.device('default.qubit', wires=2)

@qml.qnode(dev)
def circ_probs():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.probs(wires=[0, 1])

@qml.qnode(dev)
def circ_expval():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.expval(qml.PauliZ(0) @ qml.PauliZ(1))

@qml.qnode(dev)
def circ_state():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.state()

print('probs   :', circ_probs())
print('<ZZ>   :', circ_expval())
print('state   :', np.round(circ_state(), 3))

probs   : [0.5 0.  0.  0.5]
<ZZ>   : 0.9999999999999998
state   : [0.707+0.j 0.   +0.j 0.   +0.j 0.707+0.j]


Bell 상태에서 `Z⊗Z`의 기댓값은 +1이다 — 두 큐비트의 측정 결과가 완벽히 상관되어 있다는 뜻이다.

## 3.5 multiple returns
한 회로에서 여러 관측가능량을 동시에 반환할 수 있다.

In [12]:
@qml.qnode(dev)
def multi_measure():
    qml.Hadamard(wires=0)
    qml.RY(0.7, wires=1)
    qml.CNOT(wires=[0, 1])
    # 큐비트 각각의 Pauli-Z 기댓값을 동시에 반환
    return qml.expval(qml.PauliZ(0)), qml.expval(qml.PauliZ(1))

z0, z1 = multi_measure()
print(f'<Z_0> = {z0:+.4f}, <Z_1> = {z1:+.4f}')

<Z_0> = +0.0000, <Z_1> = +0.0000


## 3.6 정리
- QNode는 회로 함수와 디바이스를 묶은 호출 가능 객체다.
- 작은 회로엔 `default.qubit`, 큰 회로엔 `lightning.qubit`이 표준이다.
- `shots=None`은 정확한 기댓값, `shots=N`은 측정 표본 평균이다.
- 한 회로에서 여러 측정값을 동시에 반환할 수 있다.